# A historical heat and rainfall baseline from ERA5

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RiskThinking/crc-docs/blob/main/notebooks/sources/era5_historical_baseline.ipynb) · [GitHub preview](https://github.com/RiskThinking/crc-docs/blob/main/notebooks/sources/era5_historical_baseline.ipynb)

Build two historical baselines for six European cities from ERA5 hourly
reanalysis, 1991–2020 (the WMO climate-normal period):

- **TXx**, the hottest day of each year (annual maximum of daily maximum 2 m
  temperature), and
- **Rx1day**, the wettest day of each year (annual maximum one-day precipitation).

1. **Read annual samples.** crc-sdk's `ERA5` provider reduces hourly data to daily
   values and then to one annual extreme per 0.25° grid cell and year. Nothing is
   fitted yet, and incomplete years stay visible.
2. **Fit explicitly.** GEV by L-moments per grid cell, with a Gumbel fallback.
3. **Evaluate** return levels at the cities, and look at the thirty values behind
   each TXx curve.

**This notebook is heavy.** ERA5's public ARCO store holds one global chunk per
hour, so even a small area reads every hourly chunk: for 1991–2020 that is about
263,000 chunks per variable (about 1 TB before compression), as `inspect()`
reports below. It is not executed in CI and has no saved
outputs; run it on a machine with bandwidth and disk to spare. Later runs reuse
the cache. CRC Open (SSP5-8.5 only) has no historical baseline, which is why a
separate one like this is useful; it differs from CRC Open in source, period and
method, so their difference is not a pure climate-change signal.

In [ ]:
#@title Install dependencies and prepare this notebook
import importlib.util
import os
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB or importlib.util.find_spec("crc_sdk") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--pre",
                           "--disable-pip-version-check", "crc-sdk[zarr]>=1.0.0a3,<2", "plotly>=6"])

from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from crc_framework import DistributionFamily, SampleEstimator, SampleFit
from IPython.display import display

from crc_sdk import ERA5, Assets, Bounds, Cache, FitFailure, Period, Runtime

STATIC_PREVIEW = os.environ.get("CRC_NOTEBOOK_STATIC_PREVIEW") == "1"
pio.renderers.default = "colab" if IN_COLAB else "plotly_mimetype+png" if STATIC_PREVIEW else "plotly_mimetype"
pio.templates.default = "plotly_white"

DATA_DIR = Path("../data/era5_historical_baseline")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME = Runtime(cache=Cache.reuse(DATA_DIR / "cache"), workers=4)
CITIES = {
    "London": (-0.13, 51.51), "Paris": (2.35, 48.86), "Madrid": (-3.70, 40.42),
    "Berlin": (13.40, 52.52), "Rome": (12.50, 41.90), "Warsaw": (21.01, 52.23),
}
YEARS = (1991, 2020)
RETURN_PERIODS = [2, 10, 25]
INDICATORS = {
    "TXx": ERA5.Indicator.ANNUAL_MAXIMUM_DAILY_MAXIMUM_TEMPERATURE,
    "Rx1day": ERA5.Indicator.ANNUAL_MAXIMUM_DAILY_PRECIPITATION,
}

# Each hourly chunk is global, so one box around all six cities costs the same
# reads as a single city.
AREA = Bounds(west=-4.0, south=40.0, east=21.5, north=53.0)

## 1. Read annual extremes

One read per indicator covers all six cities. Days are UTC; hourly temperature
is instantaneous, so a day's maximum is the largest of 24 hourly values;
precipitation accumulations are assigned to the hour they end in.
`Coverage.complete()` (the default) keeps a year only if every day is complete.
`inspect()` reports the acquisition estimate before any download.

In [ ]:
annual = {}
for name, indicator in INDICATORS.items():
    recipe = ERA5(indicator=indicator).select(area=AREA, time=Period.window(*YEARS))
    for requirement in recipe.inspect(runtime=RUNTIME).requirements:
        print(f"{name}: {requirement.message}")
    annual[name] = recipe.write(DATA_DIR / f"era5_{name.lower()}.parquet", overwrite=True, runtime=RUNTIME)
    counts = annual[name].metadata.report.counts
    print(f"{name}: {counts.usable_years:,} usable and {counts.incomplete_years:,} incomplete cell-years")

## 2. Fit and evaluate

GEV by unbiased sample L-moments with a Gumbel fallback, at least 20 usable
years per grid cell. Each city is matched to the 0.25° grid cell that contains
it; ERA5's grid is an area average, so city maxima read cooler than a station.

In [ ]:
fit = SampleFit(family=DistributionFamily.GENERALIZED_EXTREME_VALUE, estimator=SampleEstimator.L_MOMENTS,
                minimum_samples=20, fallbacks=(SampleFit(family=DistributionFamily.GUMBEL_RIGHT,
                                                         minimum_samples=20),))
cities = Assets({"asset_id": list(CITIES), "longitude": [c[0] for c in CITIES.values()],
                 "latitude": [c[1] for c in CITIES.values()]})
levels = pd.concat([
    saved.fit(fit, on_failure=FitFailure.KEEP_NO_DATA).evaluate(cities).return_periods(RETURN_PERIODS)
    .read().table.to_pandas().assign(indicator=name)
    for name, saved in annual.items()
])
display(levels.pivot_table(index="asset_id", columns=["indicator", "query_value"], values="value").round(1))

## 3. The thirty values behind each TXx curve

Each city's annual TXx in its grid cell. Heat-wave years such as 2003 and
2019 stand out.

In [ ]:
fig = go.Figure()
table = annual["TXx"].read().table.to_pandas()
for city in CITIES:
    cell = levels[(levels.asset_id == city) & (levels.indicator == "TXx")].support_id.iloc[0]
    series = table[table.support_id == cell].sort_values("time")
    fig.add_trace(go.Scatter(x=series.time, y=series.value, mode="lines+markers", name=city))
fig.update_layout(title=f"ERA5 annual hottest day, {YEARS[0]}-{YEARS[1]}", xaxis_title="Year",
                  yaxis_title="TXx (°C)", width=1000, height=470)
fig.show()

## How to read this, and its limits

- **Reanalysis is not observation.** ERA5 blends observations with a weather
  model, and its biases move with the observing system.
- **Convective rainfall extremes are underestimated** by reanalysis; treat Rx1day
  as a regional baseline, not a design storm.
- **Thirty values per cell is a short record**; return levels beyond about 30
  years are extrapolations of the fitted distribution.
- **Provenance travels with the files.** Licence, attribution, checksums and the
  pinned ERA5 release are in each Parquet file's metadata; replay it offline with
  `Runtime(cache=Cache.offline(...))`.